# Export DINOv2+LoRA → ONNX (CPU)

Только экспорт с Google Drive. GPU не нужен (слабый GPU / без GPU — ок).

Выход: `pooler_output` `[B,768]` — совместимо с `DinoOnnxEncoder` в проекте.

Файлы:
- `_models/dinov2_wine_phase1.onnx`
- `_models/dinov2_wine_phase2.onnx`

In [ ]:
# 1) Drive + deps (CPU runtime достаточно)
from google.colab import drive
drive.mount("/content/drive")

%pip -q install -U "transformers>=4.40" "peft>=0.11" "torchao>=0.16" onnx onnxscript


In [ ]:
# 2) Export phase1 + phase2 best_adapter → ONNX on Drive
import os
import torch
import torch.nn as nn
from transformers import AutoModel, AutoImageProcessor
from peft import PeftModel

BASE = "/content/drive/MyDrive/ЛЦТ26/2_embed_train_data"
MODELS = os.path.join(BASE, "_models")
MODEL_NAME = "facebook/dinov2-base"
IMAGE_SIZE = 224

# что экспортировать (убери строку, если фазы нет)
ADAPTERS = [
    ("phase1", os.path.join(MODELS, "phase1", "best_adapter")),
    ("phase2", os.path.join(MODELS, "phase2", "best_adapter")),
]

class WrapPooler(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, pixel_values):
        return self.m(pixel_values=pixel_values).pooler_output

device = "cpu"  # специально CPU — GPU не трогаем
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
exported = []

for tag, adapter in ADAPTERS:
    if not os.path.isdir(adapter):
        print(f"SKIP {tag}: no adapter at {adapter}")
        continue
    print(f"=== {tag} ===")
    print("load", adapter)
    base = AutoModel.from_pretrained(MODEL_NAME)
    m = PeftModel.from_pretrained(base, adapter)
    merged = m.merge_and_unload().eval()

    out_pt = os.path.join(MODELS, f"merged_{tag}")
    os.makedirs(out_pt, exist_ok=True)
    merged.save_pretrained(out_pt)
    processor.save_pretrained(out_pt)

    onnx_path = os.path.join(MODELS, f"dinov2_wine_{tag}.onnx")
    torch.onnx.export(
        WrapPooler(merged),
        dummy,
        onnx_path,
        input_names=["pixel_values"],
        output_names=["pooler_output"],
        opset_version=17,
        dynamo=False,  # legacy exporter — no onnxscript required
        dynamic_axes={"pixel_values": {0: "B"}, "pooler_output": {0: "B"}},
    )
    mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"ONNX -> {onnx_path}  ({mb:.1f} MB)")
    exported.append(onnx_path)
    del base, m, merged

print("DONE:", exported)
print("Скачай в проект: bin/dinov2_wine_phase1.onnx  bin/dinov2_wine_phase2.onnx")
